In [1]:
import torch

from experiments_data import ExperimentLightningDataModule
from experiments import LightningOnVandermondeModel, LightningBiLipschitzAntiSymmetricModel, LightningAfaNetModel

from pytorch_lightning import callbacks, Trainer, LightningDataModule
from pytorch_lightning.loggers import TensorBoardLogger, CSVLogger
from torch.utils.data import DataLoader

import os

%load_ext autoreload
%autoreload 2

C:\Users\KalMe\anaconda3\envs\antis\Lib\inspect.py:592: UserWarning: The is_traceable field on torch.autograd.Function is deprecated and will be removed in PyTorch 2.4.
  value = getter(object, key)


In [2]:
dataset = ExperimentLightningDataModule('determinant', n_elements=2, dim=2, batch_size=128, device='cuda')#n_workers=2, persistent_workers=True)

In [3]:
dnn = LightningBiLipschitzAntiSymmetricModel(in_dim=2, in_channels=2, out_dim=1, embedding_dim=16, 
                                             optimizer_kwargs=dict(optimizer='adam', lr=0.5, amsgrad=True), 
                                             lr_scheduler_kwargs=dict(lr_scheduler='reduce', factor=0.7, patience=3, min_lr=1e-5,), 
                                             loss='l1', extra_metrics=['mse', 'l1', 'mare'], 
                                             hidden_layers=[32, 32, 32], activation='tanh', biases='all_but_last').to(device=dataset.device, dtype=dataset.dtype)

In [4]:
tb_logger = TensorBoardLogger(
    "mix_tb_logs", log_graph=True, version=f'{dnn.model_name}_{dataset.batch_size}_1'
)  # , version=version if load else None)
csv_logger = CSVLogger(
    "mix_csv_logs", version=f"{dnn.model_name}_{dataset.batch_size}_1"
)  # version=ve

In [5]:
checkpoint = callbacks.ModelCheckpoint(
    filename="best",
    monitor='val_loss',
    auto_insert_metric_name=True,
    save_top_k=1,
    save_last=True,
)
# LR_LEARN = FineTuneLearningRateFinder(k=10, strategy='mode', num_training_steps=10)

trainer = Trainer(
    logger=[csv_logger, tb_logger],
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    max_epochs=100,
    callbacks=[checkpoint, 
               callbacks.LearningRateMonitor(), 
               callbacks.RichProgressBar(leave=False), 
              ],#LR_LEARN],
    enable_checkpointing=True,
    log_every_n_steps=1, 
    # gradient_clip_val=1.0, gradient_clip_algorithm='norm'
)
trainer.fit(dnn, dataset)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┓
┃   ┃ Name          ┃ Type                          ┃ Params ┃ Mode  ┃
┡━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━┩
│ 0 │ loss          │ L1Loss                        │      0 │ train │
│ 1 │ extra_metrics │ ModuleDict                    │      0 │ train │
│ 2 │ model         │ BiLipschitzAntiSymmetricModel │  2.8 K │ train │
└───┴───────────────┴───────────────────────────────┴────────┴───────┘

Trainable params: 2.8 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 2.8 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 17                                                                                          
Modules in eval mode: 0

C:\Users\KalMe\anaconda3\envs\antis\Lib\site-packages\pytorch_lightning\loggers\tensorboard.py:195: Could not log computational graph to TensorBoard: The `model.example_input_array` attribute is not set or `input_array` was not given.


Output()

C:\Users\KalMe\anaconda3\envs\antis\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:476: 
Your `val_dataloader`'s sampler has shuffling enabled, it is strongly recommended that you turn shuffling off for 
val/test dataloaders.

C:\Users\KalMe\anaconda3\envs\antis\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: 
The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=23` in the `DataLoader` to improve performance.

C:\Users\KalMe\anaconda3\envs\antis\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: 
The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=23` in the `DataLoader` to improve performance.


Detected KeyboardInterrupt, attempting graceful shutdown ...



KeyboardInterrupt



In [3]:
%load_ext tensorboard

%tensorboard --port=6007 --logdir experiments/

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Launching TensorBoard...

In [ ]:
%load_ext tensorboard

%tensorboard --port=6007 --logdir experiments/